# CS604 最终训练：ViT-B/16 迁移到 1 万类细粒度分类（Colab 版）

这是根据短实验结果定下来的**最终训练**版本。训练代码和实验版 notebook（`cs604_vit_adaptation.ipynb`）完全一样，只是预设好了最终配置。一次「全部运行」依次完成两次训练，每次都会生成一份可以上传 CodaBench 的提交文件。

**短实验结果**（批次 `0928_0636`，A100-80GB；本地验证集为训练集每类留出 2 张，共 2 万张）：

| 方法 | 本地验证 top-1 | top-5 | 可训练参数（不含分类头） | GPU 时长 | 峰值显存 | 文档参考值 |
|---|---|---|---|---|---|---|
| 零样本（类原型） | 43.95% | 68.05% | 0 | 0.11 h | 3.9 GB | 43.7% |
| LoRA，3 epoch | 68.57% | 86.96% | 294,912（0.344%） | 0.40 h | 13.5 GB | 68.3% |
| 全量微调，3 epoch | 70.88% | 88.45% | 85,798,656（100%） | 0.45 h | 18.9 GB | 70.8% |

**据此确定的最终配置**
1. **全量微调，8 个 epoch**（CodaBench 主提交）：短实验里准确率最高。三个 epoch 的验证准确率为 68.3% → 70.6% → 70.9%，最后一个 epoch 只涨了 0.3%；而且训练 loss 在每个 epoch 开头都会下一个台阶、epoch 内几乎不降，说明模型在记忆见过的训练图（每类只有 43 张）。所以只适度加长到 8 个 epoch，不用满时间预算，以免过拟合。
2. **LoRA，10 个 epoch，分类头学习率从 1e-3 降到 2e-4**（报告里的参数高效方法）：验证准确率 62.4% → 67.0% → 68.6%，第 3 个 epoch 仍在明显上升，所以加长。另外学习率升到 1e-3 的过程中，训练 loss 反而从 2.2 升到 2.6，说明学习率偏大；全量微调里分类头用 1e-4 训练得很平稳，所以把 LoRA 的分类头学习率降到 2e-4，LoRA 模块本身保持 1e-3。这一项调整还没有单独验证过，结果以本次的本地验证和 CodaBench 分数为准。
3. **仍保留本地验证集**（每类 2 张）：用来确认加长训练没有过拟合，也为报告提供准确率曲线。

**大概要多久**：A100-80GB 上预计全量微调约 1 小时、LoRA 约 1 小时，都在单个实验 1.2 小时的预算内（96 GB·h ÷ 80 GB）；如果实际速度更慢，训练开始后会自动压缩学习率日程。整批约 2–2.5 小时（另加从 Drive 解压数据的 5–15 分钟），期间保持页面打开。

**使用方法**
1. 数据已经用 `cs604_download_to_drive.ipynb` 存在 Drive 的 `cs604/data/`。
2. `代码执行程序 → 更改运行时类型` 选 **A100 GPU**，然后 `代码执行程序 → 全部运行`，按提示授权 Google Drive。
3. 结果保存在 Drive：每个实验的提交文件和日志在 `cs604/runs/<run 名>/`，对比表和对比图在 `cs604/batches/<BATCH_NAME>/`。
4. 断线了：把 `BATCH_NAME` 设为日志开头打印的名字，再「全部运行」，已完成的实验会跳过，中断的从最近的 epoch 继续。

**几点说明**
- **训练时间预算**按显存 × 小时 = 96 GB·h 折算、每个实验单独计算：A100-80GB 为 1.2 小时，A100-40GB 为 2.4 小时。如果助教确认可以直接按 2 小时算，可以把 `TIME_BUDGET_H` 设为 2，并适当增加 `EPOCHS`。
- **参数量口径**和文档一致，不含分类头；1 万类分类头另有 7.69M 参数，单独列出。
- **基础模型**默认用文档第 1 页给的 IN21K→IN1K 版本，建议向助教确认。

## 1. 配置

In [ ]:
TEAM_ID = "06"          # 队号，提交文件名为 val_team_{TEAM_ID}.txt

# ===== 最终训练（根据批次 0928_0636 的短实验结果确定，分析见最上方说明）=====
EXPERIMENTS = [
    # 全量微调：短实验本地验证 70.88% 最好，最后一个 epoch 已接近饱和 → 适度加长到 8 个 epoch（CodaBench 主提交）
    dict(METHOD="full", EPOCHS=8),
    # LoRA：短实验 68.57%，第 3 个 epoch 仍在上升 → 加长到 10 个 epoch；
    # 学习率升到 1e-3 时 loss 反而上升 → 分类头学习率降到 2e-4（LoRA 模块保持 1e-3）
    dict(METHOD="lora", EPOCHS=10, HEAD_LR=2e-4),
]
RUN_FINAL = False       # True = 跑完上面的实验后，自动挑本地验证最好的方法、套用 FINAL_OVERRIDES 再训练一次（适合无人值守；推荐先看结果再定参数）
FINAL_METHOD = None     # None = 自动选（不含 linear）；也可以指定，比如 "lora"
FINAL_OVERRIDES = dict(EPOCHS=15, WARMUP_EPOCHS=1, HOLDOUT_PER_CLASS=0)  # 最终训练：更多轮数（超出预算会自动缩短），用满每类 45 张
BATCH_NAME = None       # None = 自动命名；断线后续跑时填上次的 BATCH_NAME
LOAD_RUN = None         # 填已训练好的 run 名：不训练，只加载权重做预测（Phase 2 用）
PREDICT_SPLITS = ["val"]  # Phase 2 发布测试集后改成 ["test"]（或 ["val", "test"]）

# ===== 默认配置（每个实验都可以在 EXPERIMENTS 里单独覆盖）=====
METHOD = "lora"         # 'lora' | 'full' | 'adaptformer' | 'adapter' | 'bitfit' | 'partial' | 'linear'
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # 文档第 1 页给的 timm 版本（IN21K→IN1K）
# MODEL_NAME = "vit_base_patch16_224.orig_in21k"            # = HF google/vit-base-patch16-224-in21k（只在 IN21K 上预训练）
HOLDOUT_PER_CLASS = 2   # 每类从训练集留出几张做本地验证（官方验证集没有标签）
EPOCHS = 10             # 最多训练几个 epoch，超出时间预算时会自动缩短
TIME_BUDGET_H = None    # None = 按 96 GB·h 自动折算；也可以直接填小时数
HEAD_INIT = "lp"        # 'lp' 线性探测 | 'proto' 类原型（零样本） | 'random'
TRAIN_HEAD = True       # PEFT 方法是否同时训练分类头

# ===== 超参数（None = 使用下面 METHOD_DEFAULTS 里的值）=====
BATCH_SIZE = None       # 单步 batch；None = 按显存自动选择
EFFECTIVE_BATCH = 256   # 等效 batch：单步放不下时用梯度累积凑到这个数
LR = None
WEIGHT_DECAY = None
HEAD_LR = None          # None = 与 LR 相同
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS = 1
RRC_SCALE = (0.25, 1.0)  # RandomResizedCrop 的面积范围
LORA_R = 8
LORA_ALPHA = 16
LORA_TARGETS = ["qkv"]   # 可选 'qkv', 'proj', 'fc1', 'fc2'
ADAPTER_DIM = 64         # AdaptFormer / Adapter 的瓶颈维度
PARTIAL_LAST_K = 2       # 'partial'：只训练最后 K 个 block
LP_EPOCHS = 10           # 线性探测的轮数（在缓存特征上训练，很快）
TTA_FLIP = False         # 预测时加水平翻转（推理延迟翻倍）
COMPILE = False          # torch.compile，A100 上一般能再快 10–20%，首次编译需要 1–2 分钟
CKPT_EVERY = 1           # 每几个 epoch 存一次断点
LOG_EVERY = 100          # 每多少步打印一次日志
SEED = 0

METHOD_DEFAULTS = {
    #                   学习率     权重衰减  逐层学习率衰减      DropPath
    "full":        dict(lr=1e-4, wd=0.05, layer_decay=0.75, drop_path=0.1),
    "partial":     dict(lr=3e-4, wd=0.05, layer_decay=1.0, drop_path=0.0),
    "lora":        dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0),
    "adaptformer": dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=0.1),
    "adapter":     dict(lr=1e-3, wd=1e-4, layer_decay=1.0, drop_path=0.0, adapter_scale=1.0),
    "bitfit":      dict(lr=1e-3, wd=0.0, layer_decay=1.0, drop_path=0.0),
    "linear":      dict(lr=0.0, wd=0.0, layer_decay=1.0, drop_path=0.0),
}
NUM_CLASSES = 10000

# ===== 路径与收尾 =====
DRIVE_DIR = "/content/drive/MyDrive/cs604"   # 结果和断点存在这里
DRIVE_DATA_DIR = f"{DRIVE_DIR}/data"         # 数据集在 Drive 上的位置（先用 cs604_download_to_drive.ipynb 存进去）
DATA_DIR = "/content/data"                   # 每个会话把数据解压到 Colab 本地磁盘，训练时从这里读图
DISCONNECT_WHEN_DONE = False  # 跑完后自动断开运行时、停止消耗计算单元（后台运行时建议 True）

## 2. 环境准备

In [ ]:
# Colab 自带 torch / torchvision，这里补装 timm 和 peft
!pip install -q timm peft
# Colab 预装的 torchao 太旧，新版 peft 创建 LoRA 时会直接报 ImportError；我们用不到 torchao，卸载即可
!pip uninstall -y -q torchao

In [ ]:
# 配置单元里的变量分两类：每个实验都能单独覆盖的（RUN_KEYS），和整批共用的（QUEUE_KEYS）
RUN_KEYS = ("METHOD", "MODEL_NAME", "HOLDOUT_PER_CLASS", "EPOCHS", "TIME_BUDGET_H", "HEAD_INIT", "TRAIN_HEAD",
            "BATCH_SIZE", "EFFECTIVE_BATCH", "LR", "WEIGHT_DECAY", "HEAD_LR", "LABEL_SMOOTHING", "WARMUP_EPOCHS",
            "RRC_SCALE", "LORA_R", "LORA_ALPHA", "LORA_TARGETS", "ADAPTER_DIM", "PARTIAL_LAST_K", "LP_EPOCHS",
            "TTA_FLIP", "COMPILE", "CKPT_EVERY", "LOG_EVERY", "SEED", "METHOD_DEFAULTS", "NUM_CLASSES")
QUEUE_KEYS = ("TEAM_ID", "EXPERIMENTS", "RUN_FINAL", "FINAL_METHOD", "FINAL_OVERRIDES", "BATCH_NAME", "LOAD_RUN",
              "PREDICT_SPLITS", "DRIVE_DIR", "DRIVE_DATA_DIR", "DATA_DIR", "DISCONNECT_WHEN_DONE")
BASE_CONFIG = {k: globals()[k] for k in RUN_KEYS}
QUEUE_CONFIG = {k: globals()[k] for k in QUEUE_KEYS}

import os, re, gc, sys, json, math, time, types, random, shutil, zipfile, subprocess, contextlib
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules

# ---- 挂载 Google Drive：数据从这里读，结果和断点也存在这里，断线不会丢 ----
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = DRIVE_DIR
else:
    OUT_ROOT = os.path.abspath("outputs")

# ---- 设备与混合精度 ----
if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    gib = props.total_memory / 1024**3
    VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # 标称显存
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4/V100 没有 bf16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
    GPU_NAME, VRAM_GB, AMP_DTYPE = DEVICE, None, None
AMP_NAME = str(AMP_DTYPE).replace("torch.", "") if AMP_DTYPE else "fp32"
NUM_WORKERS = min(16, os.cpu_count() or 2)

BATCH_NAME = BATCH_NAME or datetime.now().strftime("%m%d_%H%M")
BATCH_DIR = os.path.join(OUT_ROOT, "batches", BATCH_NAME)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
with open(os.path.join(BATCH_DIR, "queue_config.json"), "w") as f:
    json.dump({**QUEUE_CONFIG, "BATCH_NAME": BATCH_NAME, "BASE_CONFIG": BASE_CONFIG}, f, indent=2)

LOG_FILE = None  # 当前实验的 train_log.txt，由 run_experiment 设置


def log(*args):
    """打印，同时追加到这批实验的 batch_log.txt 和当前实验的 train_log.txt（报告要附训练日志）"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    stamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    for path in (os.path.join(BATCH_DIR, "batch_log.txt"), LOG_FILE):
        if path:
            with open(path, "a") as f:
                f.write(stamp + msg + "\n")


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Batch {BATCH_NAME} -> {BATCH_DIR}  (断线后续跑：把 BATCH_NAME 设为 \"{BATCH_NAME}\")")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), AMP {AMP_NAME}, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
if AMP_DTYPE == torch.float16:
    log("WARNING: this GPU has no bf16 (e.g. T4); using fp16 + GradScaler, training will be slow.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. 从 Drive 准备数据

- 数据只从 Drive 的 `DRIVE_DATA_DIR`（默认 `cs604/data/`）读取，需要先用 `cs604_download_to_drive.ipynb` 存进去。找不到时会直接报错提示。
- 每个新会话会把 Drive 上的压缩包解压到 Colab 本地磁盘（`DATA_DIR`），训练时从本地读图：直接从 Drive 逐张读取 45 万张小图会非常慢。
- 图片保持原图（最长边 500px），不预先缩小：A100 运行时的 CPU 解码速度足够跟上 GPU。
- 解压完成后会写一个标记文件，同一个 Colab 会话里重复运行会直接跳过。

In [ ]:
def prepare_split(split):
    """把 Drive 上的 {split}.tar.gz 解压到 Colab 本地磁盘 DATA_DIR/{split}/，并核对图片数量"""
    list_path = os.path.join(DRIVE_DATA_DIR, f"{split}.txt")
    tar_path = os.path.join(DRIVE_DATA_DIR, f"{split}.tar.gz")
    missing = [p for p in (list_path, tar_path) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Drive 上找不到 {missing}：请先运行 cs604_download_to_drive.ipynb 把数据存进 Drive"
                                "（Phase 2 的测试集发布后要再运行一次）")
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if not os.path.exists(marker):
        t0 = time.time()
        log(f"[{split}] extracting {tar_path} ...")
        subprocess.run(["tar", "-xzf", tar_path, "-C", DATA_DIR], check=True)
        n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
        assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
        open(marker, "w").close()
        log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


def read_list(path):
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1]) if len(parts) > 1 else -1))
    return items


TRAIN_ALL = [] if LOAD_RUN else read_list(prepare_split("train"))  # 只做预测时不需要训练集
PRED_ITEMS = {split: read_list(prepare_split(split)) for split in PREDICT_SPLITS}
log(f"train {len(TRAIN_ALL):,} images | " + " | ".join(f"{s} {len(v):,}" for s, v in PRED_ITEMS.items())
    + f" | {NUM_WORKERS} loader workers")

## 4. 模型与微调方法

- `lora` 用 Hugging Face [PEFT](https://github.com/huggingface/peft) 实现，训练完会把 LoRA 权重合并回原矩阵，推理时没有额外开销。
- `adaptformer` 按 [AdaptFormer 官方实现](https://github.com/ShoufaChen/AdaptFormer)：瓶颈 64，缩放 0.1，与 FFN 并联，上投影零初始化。
- `adapter` 是串行 Adapter，接在 FFN 之后。
- `bitfit` 只训练 bias，`partial` 只训练最后 `PARTIAL_LAST_K` 个 block，`linear` 只用分类头。

ViT-B/16 上各方法的可训练参数（默认配置，不含分类头；backbone 共 85,798,656 个参数）：

| 方法 | 可训练参数 | 占 backbone | 推理时额外参数 |
|---|---|---|---|
| `full` | 85,798,656 | 100% | 0 |
| `lora`（r=8，qkv） | 294,912 | 0.344% | 0（已合并） |
| `adaptformer` / `adapter`（瓶颈 64） | 1,189,632 | 1.387% | 1,189,632 |
| `bitfit` | 102,912 | 0.120% | 0 |
| `partial`（最后 2 个 block） | 14,177,280 | 16.5%（超过文档建议的 5%） | 0 |
| `linear` | 0 | 0% | 0 |

1 万类分类头另有 7,690,000 个参数，报告里需要单独说明它是否参与训练。

In [ ]:
class Adapter(nn.Module):
    """瓶颈 MLP：down → ReLU → up。up 零初始化，所以插入后模型输出一开始不变（同 AdaptFormer 官方实现）"""

    def __init__(self, dim, bottleneck, scale, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        self.drop = nn.Dropout(dropout)
        self.scale = scale
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.drop(F.relu(self.down(x)))) * self.scale


def adaptformer_block_forward(self, x, *args, **kwargs):
    # AdaptFormer：adapter 与 FFN 并联，输入是进入 FFN 分支之前的残差流
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    return x + self.drop_path2(self.ls2(self.mlp(self.norm2(x)))) + self.adapter(x)


def adapter_block_forward(self, x, *args, **kwargs):
    # 串行 Adapter：接在 FFN 输出之后，自带残差
    x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x), *args, **kwargs)))
    h = self.mlp(self.norm2(x))
    return x + self.drop_path2(self.ls2(h + self.adapter(h)))


LORA_MODULES = {"qkv": r"attn\.qkv", "proj": r"attn\.proj", "fc1": r"mlp\.fc1", "fc2": r"mlp\.fc2"}


def is_head(name):
    return re.search(r"(^|\.)head\.(weight|bias)$", name) is not None


def unwrap(net):
    return net.get_base_model() if hasattr(net, "get_base_model") else net


def resolve_config(overrides):
    """默认配置 + 单个实验的覆盖项 → 这次实验的完整配置"""
    unknown = set(overrides) - set(RUN_KEYS)
    assert not unknown, f"unknown keys in experiment config: {sorted(unknown)}"
    cfg = SimpleNamespace(**{**BASE_CONFIG, **overrides})
    defaults = cfg.METHOD_DEFAULTS[cfg.METHOD]
    cfg.LR = defaults["lr"] if cfg.LR is None else cfg.LR
    cfg.WEIGHT_DECAY = defaults["wd"] if cfg.WEIGHT_DECAY is None else cfg.WEIGHT_DECAY
    cfg.HEAD_LR = cfg.LR if cfg.HEAD_LR is None else cfg.HEAD_LR
    cfg.LAYER_DECAY, cfg.DROP_PATH = defaults["layer_decay"], defaults["drop_path"]
    cfg.ADAPTER_SCALE = defaults.get("adapter_scale", 1.0)
    if cfg.BATCH_SIZE is None:
        cfg.BATCH_SIZE = 256 if (VRAM_GB or 0) >= 40 else 128 if (VRAM_GB or 0) >= 24 else 64
    cfg.ACCUM_STEPS = max(1, cfg.EFFECTIVE_BATCH // cfg.BATCH_SIZE)
    if cfg.TIME_BUDGET_H is None:
        cfg.TIME_BUDGET_H = 96 / VRAM_GB if VRAM_GB else 1.0  # 2 h × 48 GB = 96 GB·h
    return cfg


def create_model(cfg):
    return timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=cfg.NUM_CLASSES,
                             drop_path_rate=cfg.DROP_PATH).to(DEVICE)


def apply_method(net, cfg):
    """按 METHOD 冻结参数、插入 PEFT 模块。返回的模型可能被 peft 包了一层"""
    if cfg.METHOD == "full":
        return net.requires_grad_(True)
    net.requires_grad_(False)
    if cfg.METHOD == "lora":
        from peft import LoraConfig, get_peft_model
        pattern = r"blocks\.\d+\.(" + "|".join(LORA_MODULES[t] for t in cfg.LORA_TARGETS) + ")"
        net = get_peft_model(net, LoraConfig(r=cfg.LORA_R, lora_alpha=cfg.LORA_ALPHA, lora_dropout=0.0,
                                             target_modules=pattern, bias="none"))
    elif cfg.METHOD in ("adaptformer", "adapter"):
        forward = adaptformer_block_forward if cfg.METHOD == "adaptformer" else adapter_block_forward
        for blk in net.blocks:
            blk.adapter = Adapter(net.embed_dim, cfg.ADAPTER_DIM, cfg.ADAPTER_SCALE)
            blk.forward = types.MethodType(forward, blk)
    elif cfg.METHOD == "bitfit":
        for n, p in net.named_parameters():
            if n.endswith(".bias"):
                p.requires_grad_(True)
    elif cfg.METHOD == "partial":
        for blk in net.blocks[-cfg.PARTIAL_LAST_K:]:
            blk.requires_grad_(True)
        net.norm.requires_grad_(True)
    unwrap(net).head.requires_grad_(cfg.TRAIN_HEAD and cfg.METHOD != "linear")
    return net


def param_report(net, cfg, ctx, head_stats):
    named = list(net.named_parameters())
    head_trained = any(p.requires_grad for n, p in named if is_head(n))
    n_train = sum(p.numel() for n, p in named if p.requires_grad and not is_head(n))
    source = head_stats.get("head_source", cfg.HEAD_INIT)
    head_mode = ("fine-tuned" if head_trained else "linear probe" if source == "lp"
                 else "class prototypes (not trained)" if source == "proto" else "random, frozen")
    return {"backbone_params": ctx.n_backbone, "head_params": ctx.n_head, "head_mode": head_mode,
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / ctx.n_backbone,
            "trainable_incl_head": n_train + (ctx.n_head if head_trained or source == "lp" else 0)}

## 5. 数据集与本地验证集

官方验证集没有标签，所以从训练集每类留出 `HOLDOUT_PER_CLASS` 张图作本地验证集，用来调参和做消融实验。划分由 `SEED` 固定，每次运行都一样。

In [ ]:
def split_holdout(items, k, seed):
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


BICUBIC = T.InterpolationMode.BICUBIC


def make_ctx(model, cfg):
    """这次实验的数据设置：训练/本地验证划分、图像变换、归一化参数、参数量基数"""
    data_cfg = timm.data.resolve_model_data_config(model)
    size = data_cfg["input_size"][-1]
    train_items, holdout_items = split_holdout(TRAIN_ALL, cfg.HOLDOUT_PER_CLASS, cfg.SEED) if TRAIN_ALL else ([], [])
    return SimpleNamespace(
        data_cfg=data_cfg, img_size=size, train_items=train_items, holdout_items=holdout_items,
        mean=torch.tensor(data_cfg["mean"], device=DEVICE).view(1, 3, 1, 1) * 255,
        std=torch.tensor(data_cfg["std"], device=DEVICE).view(1, 3, 1, 1) * 255,
        # 在 CPU 上只做裁剪/翻转，输出 uint8；归一化放到 GPU 上做，减轻 CPU 负担
        train_tf=T.Compose([T.RandomResizedCrop(size, scale=tuple(cfg.RRC_SCALE), interpolation=BICUBIC),
                            T.RandomHorizontalFlip(), T.PILToTensor()]),
        eval_tf=T.Compose([T.Resize(int(size / data_cfg["crop_pct"]), interpolation=BICUBIC),
                           T.CenterCrop(size), T.PILToTensor()]),
        n_backbone=sum(p.numel() for n, p in model.named_parameters() if not is_head(n)),
        n_head=sum(p.numel() for p in model.head.parameters()),
    )


class Loader(DataLoader):
    """每次开新的 worker 进程前，先在主进程回收垃圾。否则中断过的旧 DataLoader 会被复制进新 worker，
    在那里被回收时刷出一大片无害的 AssertionError: can only test a child process。"""

    def __iter__(self):
        if self.num_workers > 0:
            gc.collect()
        return super().__iter__()


def make_loader(items, train, ctx, cfg, shuffle=False):
    return Loader(ImageList(items, ctx.train_tf if train else ctx.eval_tf),
                  batch_size=cfg.BATCH_SIZE if train else 2 * cfg.BATCH_SIZE, shuffle=shuffle, drop_last=train,
                  num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda",
                  persistent_workers=train and NUM_WORKERS > 0)


def to_input(x, ctx):
    return (x.to(DEVICE, non_blocking=True).float() - ctx.mean) / ctx.std


def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE == "cuda" else contextlib.nullcontext()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def free_gpu():
    """释放上一个实验留下的显存，并重置显存峰值统计"""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

## 6. 分类头初始化

对应文档提示里的 *Initialization*。新的 1 万类分类头如果随机初始化，训练初期的梯度会破坏预训练特征。这里先用冻结的 backbone 提取一遍训练集特征：

1. **类原型**：每类特征均值（L2 归一化）作为分类头权重，这就是文档里的 "Zero-Shot with initialization of classifier head"（0 个可训练参数）。
2. **线性探测**：以原型为起点，在缓存的特征上训练分类头，几秒钟就能完成。如果在本地验证集上不如原型，就保留原型。

同一批实验里，只要模型和数据划分相同，特征只提取一次、分类头只算一次，后面的实验直接复用；但**每个实验的 GPU 时长都会计入这部分耗时**，保证报告里的时间如实。续训或只做预测时跳过这一步。

In [ ]:
LP_LR, LP_WD, LP_BATCH = 1e-3, 1e-4, 4096
FEATURE_CACHE, HEAD_CACHE = {}, {}


class Clock:
    """累计 GPU 工作时间（分类头初始化 + 训练 + 训练中的验证），不含数据准备和存断点"""

    def __init__(self):
        self.used, self._t0 = 0.0, None

    def __enter__(self):
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        sync()
        self.used += time.time() - self._t0
        self._t0 = None

    @property
    def elapsed(self):
        return self.used + (time.time() - self._t0 if self._t0 else 0.0)


@torch.inference_mode()
def extract_features(net, items, ctx, cfg, desc):
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, False, ctx, cfg), desc=desc, leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x, ctx)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels, num_classes):
    protos = torch.zeros(num_classes, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """给余弦相似度挑一个合适的温度，让原型头的 softmax 不至于太平或太尖"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


@torch.inference_mode()
def topk_accuracy(logits_fn, feats, labels, chunk=4096):
    c1 = c5 = 0
    for i in range(0, len(feats), chunk):
        top5 = logits_fn(feats[i:i + chunk].float()).topk(5, dim=1).indices
        y = labels[i:i + chunk]
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return c1 / len(feats), c5 / len(feats)


def linear_probe(feats, labels, weight, bias, cfg):
    head = nn.Linear(feats.shape[1], cfg.NUM_CLASSES).to(DEVICE)
    with torch.no_grad():
        head.weight.copy_(weight)
        head.bias.copy_(bias)
    opt = torch.optim.AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WD)
    total = cfg.LP_EPOCHS * math.ceil(len(feats) / LP_BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / total)))
    for _ in tqdm(range(cfg.LP_EPOCHS), desc="linear probe", leave=False):
        perm = torch.randperm(len(feats), device=feats.device)
        for i in range(0, len(feats), LP_BATCH):
            idx = perm[i:i + LP_BATCH]
            loss = F.cross_entropy(head(feats[idx].float()), labels[idx], label_smoothing=cfg.LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
    return head.weight.detach(), head.bias.detach()


def cached_features(model, ctx, cfg):
    """冻结 backbone 的训练集 / 本地验证集特征；同一批实验里相同的模型和划分只提取一次"""
    key = (cfg.MODEL_NAME, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    if key in FEATURE_CACHE:
        log("Reusing cached backbone features from an earlier experiment in this batch")
        return FEATURE_CACHE[key]
    FEATURE_CACHE.clear()  # 只保留一份，节省内存
    t0 = time.time()
    f_tr, y_tr = extract_features(model, ctx.train_items, ctx, cfg, "train features")
    f_ho, y_ho = (extract_features(model, ctx.holdout_items, ctx, cfg, "holdout features")
                  if ctx.holdout_items else (None, None))
    sync()
    FEATURE_CACHE[key] = dict(f_tr=f_tr.cpu(), y_tr=y_tr.cpu(), seconds=time.time() - t0,
                              f_ho=None if f_ho is None else f_ho.cpu(), y_ho=None if y_ho is None else y_ho.cpu())
    log(f"Extracted features for {len(f_tr):,} train / {len(ctx.holdout_items):,} holdout images "
        f"in {FEATURE_CACHE[key]['seconds'] / 60:.1f} min")
    return FEATURE_CACHE[key]


def init_head(model, ctx, cfg):
    """初始化分类头，返回 (head_stats, 耗时秒数)。耗时包含提取特征的时间，复用缓存时也照算"""
    key = (cfg.MODEL_NAME, cfg.HOLDOUT_PER_CLASS, cfg.SEED, cfg.HEAD_INIT, cfg.LP_EPOCHS, cfg.LABEL_SMOOTHING)
    if key not in HEAD_CACHE:
        feats = cached_features(model, ctx, cfg)
        t0 = time.time()
        f_tr, y_tr = feats["f_tr"].to(DEVICE), feats["y_tr"].to(DEVICE)
        protos = class_prototypes(f_tr, y_tr, cfg.NUM_CLASSES)
        scale = fit_logit_scale(protos, f_tr, y_tr)
        # 线性头作用在未归一化的特征上：除以平均特征模长，让 logits ≈ scale × 余弦相似度
        weight = protos * (scale / f_tr.float().norm(dim=1).mean())
        bias = torch.zeros(cfg.NUM_CLASSES, device=DEVICE)
        stats = dict(head_source="proto", proto_logit_scale=scale)
        if feats["f_ho"] is not None:
            f_ho, y_ho = feats["f_ho"].to(DEVICE), feats["y_ho"].to(DEVICE)
            stats["proto_top1"], stats["proto_top5"] = topk_accuracy(lambda f: f @ weight.T, f_ho, y_ho)
            log(f"Prototype head (zero-shot, 0 trainable params): holdout top-1 {stats['proto_top1']:.2%}")
        if cfg.HEAD_INIT == "lp":
            lp_w, lp_b = linear_probe(f_tr, y_tr, weight, bias, cfg)
            if feats["f_ho"] is not None:
                stats["lp_top1"], stats["lp_top5"] = topk_accuracy(lambda f: f @ lp_w.T + lp_b, f_ho, y_ho)
                log(f"Linear-probe head: holdout top-1 {stats['lp_top1']:.2%}")
            if feats["f_ho"] is None or stats["lp_top1"] >= stats["proto_top1"]:
                weight, bias = lp_w, lp_b
                stats["head_source"] = "lp"
            else:
                log("Linear probe is worse than the prototypes on holdout; keeping the prototype head")
        sync()
        HEAD_CACHE[key] = dict(weight=weight.cpu(), bias=bias.cpu(), stats=stats,
                               seconds=feats["seconds"] + time.time() - t0)
    else:
        log("Reusing the cached classifier head from an earlier experiment in this batch")
    cached = HEAD_CACHE[key]
    with torch.no_grad():
        model.head.weight.copy_(cached["weight"].to(DEVICE))
        model.head.bias.copy_(cached["bias"].to(DEVICE))
    log(f"Head initialized from {cached['stats']['head_source']} "
        f"({cached['seconds'] / 60:.1f} min incl. feature extraction, counted in this run's GPU time)")
    return dict(cached["stats"]), cached["seconds"]

## 7. 训练

- AdamW，前 `WARMUP_EPOCHS` 个 epoch 线性 warmup，之后余弦衰减。bf16 混合精度，梯度裁剪 1.0，label smoothing 0.1。
- 前 120 步测速后，如果按 `EPOCHS` 跑会超出时间预算，就把学习率日程压缩到预算内能跑完的步数。
- 每个 epoch 结束时在本地验证集上评估一次，并把断点存到 Drive。

In [ ]:
def build_param_groups(net, cfg):
    """分类头用 HEAD_LR；其余参数按层衰减学习率（只有 full 默认开启）；bias、norm、位置编码不做 weight decay"""
    num_layers = len(unwrap(net).blocks) + 1
    groups = {}
    for n, p in net.named_parameters():
        if not p.requires_grad:
            continue
        name = n.replace("base_model.model.", "")
        if is_head(name):
            lr = cfg.HEAD_LR
        else:
            m = re.match(r"blocks\.(\d+)\.", name)
            if m:
                layer = int(m.group(1)) + 1
            elif name.startswith(("cls_token", "pos_embed", "reg_token", "patch_embed")):
                layer = 0
            else:
                layer = num_layers
            lr = cfg.LR * cfg.LAYER_DECAY ** (num_layers - layer)
        wd = 0.0 if p.ndim <= 1 or name in ("cls_token", "pos_embed", "reg_token") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd, "base_lr": lr} for (lr, wd), ps in groups.items()]


@torch.inference_mode()
def evaluate(net, loader, ctx):
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x, ctx)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_state(net, path, **extra):
    """只存可训练参数和分类头（LoRA 约 30MB，full 约 370MB）"""
    state = {n: p.detach().cpu() for n, p in net.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def train(model, ctx, cfg, run_dir, clock, head_stats, resume_ckpt):
    """训练并返回 (hist, step, steps_per_epoch, head_stats)；linear 方法不训练，直接返回"""
    spe = len(ctx.train_items) // (cfg.BATCH_SIZE * cfg.ACCUM_STEPS)
    sched = {"total": 0 if cfg.METHOD == "linear" else cfg.EPOCHS * spe,
             "warmup": max(1, int(cfg.WARMUP_EPOCHS * spe))}
    hist = {"step": [], "loss": [], "lr": [], "epoch": [], "epoch_loss": [], "holdout_top1": [], "steps_per_epoch": spe}
    step, start_epoch = 0, 0
    if not sched["total"]:
        return hist, step, spe, head_stats

    def lr_factor(s):
        if s < sched["warmup"]:
            return (s + 1) / sched["warmup"]
        progress = (s - sched["warmup"]) / max(1, sched["total"] - sched["warmup"])
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(build_param_groups(model, cfg), **({"fused": True} if DEVICE == "cuda" else {}))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    if resume_ckpt:
        ck = torch.load(resume_ckpt, map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        step, start_epoch = ck["step"], ck["epoch"] + 1
        sched.update(total=ck["total_steps"], warmup=ck["warmup_steps"])
        clock.used, hist, head_stats = ck["clock"], ck["hist"], ck["head_stats"]
        log(f"Resumed at epoch {start_epoch + 1}, step {step}, {clock.used / 3600:.2f} GPU-hours already used")

    net = torch.compile(model) if cfg.COMPILE and DEVICE == "cuda" else model
    train_loader = make_loader(ctx.train_items, True, ctx, cfg, shuffle=True)
    holdout_loader = make_loader(ctx.holdout_items, False, ctx, cfg) if ctx.holdout_items else None
    budget_s = cfg.TIME_BUDGET_H * 3600
    log(f"Training up to {cfg.EPOCHS} epochs x {spe} steps (batch {cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}, "
        f"warmup {sched['warmup']} steps)")
    first_step, t_mark, t_start, done = step, 0.0, clock.elapsed, step >= sched["total"]
    for epoch in range(start_epoch, cfg.EPOCHS):
        if done:
            break
        net.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, run_n, ep_loss, ep_n, t_log = 0.0, 0, 0.0, 0, time.time()
        with clock:
            for i, (x, y) in enumerate(train_loader):
                with autocast():
                    loss = F.cross_entropy(net(to_input(x, ctx)), y.to(DEVICE, non_blocking=True),
                                           label_smoothing=cfg.LABEL_SMOOTHING)
                scaler.scale(loss / cfg.ACCUM_STEPS).backward()
                run_loss += loss.detach()
                run_n += 1
                if (i + 1) % cfg.ACCUM_STEPS:
                    continue
                for g in optimizer.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

                # 测速：按实际速度把学习率日程压缩到时间预算以内
                if step - first_step == 20:
                    t_mark = clock.elapsed
                elif step - first_step == 120:
                    sec_per_step = (clock.elapsed - t_mark) / 100
                    fit = step + int((0.97 * budget_s - clock.elapsed - 30 * (cfg.EPOCHS - epoch)) / sec_per_step)
                    log(f"Speed {cfg.BATCH_SIZE * cfg.ACCUM_STEPS / sec_per_step:.0f} img/s, "
                        f"{sec_per_step * spe / 60:.1f} min/epoch")
                    if fit < sched["total"]:
                        sched["total"] = max(fit, step + 1)
                        sched["warmup"] = min(sched["warmup"], max(1, sched["total"] // 10))
                        log(f"Time budget fits ~{sched['total'] / spe:.1f} epochs: "
                            f"LR schedule compressed to {sched['total']} steps")

                if step % cfg.LOG_EVERY == 0:
                    avg = (run_loss / run_n).item()
                    if not math.isfinite(avg):
                        raise RuntimeError("Loss is NaN/inf; try a lower LR")
                    eta = (sched["total"] - step) * (clock.elapsed - t_start) / (step - first_step)
                    log(f"ep {epoch + 1}/{cfg.EPOCHS} step {step}/{sched['total']} | loss {avg:.4f} | "
                        f"lr {cfg.LR * lr_factor(step):.2e} | {run_n * cfg.BATCH_SIZE / (time.time() - t_log):.0f} img/s | "
                        f"{clock.elapsed / 60:.1f} min | ETA {eta / 60:.0f} min")
                    hist["step"].append(step)
                    hist["loss"].append(avg)
                    hist["lr"].append(cfg.LR * lr_factor(step))
                    ep_loss += avg * run_n
                    ep_n += run_n
                    run_loss, run_n, t_log = 0.0, 0, time.time()
                if step >= sched["total"] or clock.elapsed > 1.05 * budget_s:
                    done = True
                    break
            if run_n:
                ep_loss += float(run_loss)
                ep_n += run_n
            top1 = evaluate(net, holdout_loader, ctx)[0] if holdout_loader else None
        hist["epoch"].append(epoch + 1)
        hist["epoch_loss"].append(ep_loss / max(1, ep_n))
        hist["holdout_top1"].append(top1)
        log(f"== epoch {epoch + 1}: train loss {ep_loss / max(1, ep_n):.4f}"
            + (f", holdout top-1 {top1:.2%}" if top1 is not None else "") + f", {clock.elapsed / 60:.1f} min used")
        if (epoch + 1) % cfg.CKPT_EVERY == 0 or done or epoch + 1 == cfg.EPOCHS:
            save_state(model, os.path.join(run_dir, "last.pt"), optimizer=optimizer.state_dict(),
                       scaler=scaler.state_dict(), step=step, epoch=epoch, total_steps=sched["total"],
                       warmup_steps=sched["warmup"], clock=clock.used, hist=hist, head_stats=head_stats)
    return hist, step, spe, head_stats

## 8. 评估、推理延迟与提交文件

推理延迟用随机输入测量纯模型前向时间（不含图片解码），与训练时使用相同的混合精度。报告里请写明 GPU 型号、batch size 和精度。开启 `TTA_FLIP` 时，单张延迟按两次前向计算。

In [ ]:
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # 分类色板，按固定顺序使用


def style_axis(ax, title, xlabel, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11)
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


def plot_run_curves(hist, cfg, run_dir):
    if not hist["step"]:
        return
    accs = [(e, 100 * a) for e, a in zip(hist["epoch"], hist["holdout_top1"]) if a is not None]
    fig, axes = plt.subplots(1, 2 if accs else 1, figsize=(11 if accs else 5.5, 3.6), facecolor=SURFACE, squeeze=False)
    ax = axes[0][0]
    ax.plot(hist["step"], hist["loss"], color=SERIES[0], linewidth=1.5, solid_capstyle="round")
    style_axis(ax, f"Training loss ({cfg.METHOD})", "optimizer step", f"cross-entropy (label smoothing {cfg.LABEL_SMOOTHING})")
    if accs:
        ax = axes[0][1]
        ep, acc = zip(*accs)
        ax.plot(ep, acc, color=SERIES[0], linewidth=1.5, marker="o", markersize=6,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax.annotate(f"{acc[-1]:.1f}%", (ep[-1], acc[-1]), textcoords="offset points", xytext=(0, 8),
                    ha="center", color=INK, fontsize=9)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        style_axis(ax, "Holdout top-1 accuracy", "epoch", "accuracy (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, "curves.png"), dpi=150, facecolor=SURFACE)
    plt.close(fig)


@torch.inference_mode()
def measure_latency(net, batch_size, iters, ctx, warmup=10):
    net.eval()
    x = torch.randn(batch_size, 3, ctx.img_size, ctx.img_size, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    sync()
    return (time.perf_counter() - t0) / iters


def latency_report(net, ctx, cfg):
    big, k = (256 if DEVICE == "cuda" else 8), (2 if cfg.TTA_FLIP else 1)
    lat_1 = measure_latency(net, 1, 100 if DEVICE == "cuda" else 5, ctx) * k
    lat_b = measure_latency(net, big, 20 if DEVICE == "cuda" else 2, ctx) * k
    setting = f"{GPU_NAME}, {AMP_NAME}, {'with' if cfg.TTA_FLIP else 'no'} flip TTA"
    log(f"Inference latency (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch {big}): {big / lat_b:.0f} img/s | {setting}")
    return dict(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=big / lat_b, latency_setting=setting)


@torch.inference_mode()
def predict(net, items, ctx, cfg):
    net.eval()
    preds = []
    for x, _ in tqdm(make_loader(items, False, ctx, cfg), desc="predict", leave=False):
        x = to_input(x, ctx)
        with autocast():
            logits = net(x).float()
            if cfg.TTA_FLIP:
                logits += net(torch.flip(x, dims=[3])).float()
        preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).tolist()


def make_submission(split, items, net, ctx, cfg, run_name, run_dir):
    """预测一个数据集并写出 CodaBench 提交文件：{split}_team_XX.txt 打包成 zip"""
    preds = predict(net, items, ctx, cfg)
    assert len(preds) == len(items) == len({p for p, _ in items}), "every image must appear exactly once"
    assert all(0 <= c < cfg.NUM_CLASSES for c in preds)
    name = f"{split}_team_{TEAM_ID}.txt"
    txt_path = os.path.join(run_dir, name)
    with open(txt_path, "w") as f:
        f.writelines(f"{path} {c}\n" for (path, _), c in zip(items, preds))
    zip_path = os.path.join(run_dir, f"submission_{split}_{run_name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(txt_path, arcname=name)
    counts = np.bincount(preds, minlength=cfg.NUM_CLASSES)
    log(f"[{split}] {len(preds):,} predictions over {np.count_nonzero(counts):,} classes "
        f"(max {counts.max()} per class) -> {zip_path}")
    return zip_path

## 9. 单个实验的完整流程

`run_experiment` 依次完成：建模型 → 初始化分类头 → 插入微调模块 → 训练 → 本地评估 → 测推理延迟 → 生成提交文件 → 写 `summary.json`。每个实验的结果都在 Drive 的 `cs604/runs/<run 名>/` 里：`train_log.txt`、`config.json`、`summary.json`、`history.json`、`curves.png`、提交 zip 和权重。

In [ ]:
ARCH_KEYS = ("METHOD", "MODEL_NAME", "NUM_CLASSES", "HEAD_INIT", "TRAIN_HEAD", "LORA_R", "LORA_ALPHA",
             "LORA_TARGETS", "ADAPTER_DIM", "PARTIAL_LAST_K", "METHOD_DEFAULTS")


def run_experiment(overrides, run_name):
    """跑完一个实验并返回结果汇总；已完成的实验直接读取结果，中断的实验从断点继续"""
    global LOG_FILE
    cfg = resolve_config(overrides)
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    summary_path = os.path.join(run_dir, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            previous = json.load(f)
        if all(s in previous.get("submissions", {}) for s in PREDICT_SPLITS):
            log(f"== {run_name}: already finished, skipping")
            return previous
    os.makedirs(run_dir, exist_ok=True)
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({**vars(cfg), "RUN_NAME": run_name, "BATCH_NAME": BATCH_NAME, "overrides": overrides}, f, indent=2)
        log(f"\n========== {run_name} ==========")
        log(f"{cfg.METHOD}: lr {cfg.LR}, head lr {cfg.HEAD_LR}, wd {cfg.WEIGHT_DECAY}, layer decay {cfg.LAYER_DECAY}, "
            f"epochs {cfg.EPOCHS}, holdout {cfg.HOLDOUT_PER_CLASS}/class, head init {cfg.HEAD_INIT}, "
            f"time budget {cfg.TIME_BUDGET_H:.2f} h")
        random.seed(cfg.SEED)
        np.random.seed(cfg.SEED)
        torch.manual_seed(cfg.SEED)
        free_gpu()

        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        clock = Clock()
        resume_ckpt = os.path.join(run_dir, "last.pt")
        resume_ckpt = resume_ckpt if os.path.exists(resume_ckpt) else None
        head_stats = {}
        if resume_ckpt:
            log("Found last.pt: resuming this run (head initialization skipped)")
        elif cfg.HEAD_INIT != "random":
            head_stats, seconds = init_head(model, ctx, cfg)
            clock.used += seconds
        model = apply_method(model, cfg).to(DEVICE)
        p0 = param_report(model, cfg, ctx, head_stats)
        log(f"Trainable (excl. head): {p0['trainable_excl_head']:,} = {p0['trainable_pct']:.3f}% of backbone | "
            f"head {ctx.n_head:,} params ({p0['head_mode']})")

        hist, step, spe, head_stats = train(model, ctx, cfg, run_dir, clock, head_stats, resume_ckpt)
        save_state(model, os.path.join(run_dir, "final.pt"), config=vars(cfg), head_stats=head_stats)
        with open(os.path.join(run_dir, "history.json"), "w") as f:
            json.dump(hist, f)
        log(f"Training done: {step} steps, {clock.elapsed / 3600:.2f} GPU-hours (budget {cfg.TIME_BUDGET_H:.2f} h)")

        params = param_report(model, cfg, ctx, head_stats)
        if cfg.METHOD == "lora":
            model = model.merge_and_unload()  # LoRA 权重并回原矩阵，推理时没有额外开销
        params["extra_params_at_inference"] = sum(p.numel() for p in model.parameters()) - ctx.n_backbone - ctx.n_head
        results = {}
        if ctx.holdout_items:
            results["holdout_top1"], results["holdout_top5"] = evaluate(
                model, make_loader(ctx.holdout_items, False, ctx, cfg), ctx)
            log(f"Final holdout: top-1 {results['holdout_top1']:.2%}, top-5 {results['holdout_top5']:.2%}")
        plot_run_curves(hist, cfg, run_dir)
        results.update(latency_report(model, ctx, cfg))
        submissions = {split: make_submission(split, items, model, ctx, cfg, run_name, run_dir)
                       for split, items in PRED_ITEMS.items()}

        summary = dict(run=run_name, method=cfg.METHOD, model=cfg.MODEL_NAME, overrides=overrides, **params,
                       **results, **head_stats, gpu=GPU_NAME, vram_gb=VRAM_GB, amp=AMP_NAME,
                       batch=f"{cfg.BATCH_SIZE}x{cfg.ACCUM_STEPS}", epochs_trained=round(step / max(1, spe), 2),
                       steps=step, gpu_hours=clock.elapsed / 3600, budget_hours=cfg.TIME_BUDGET_H,
                       peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None,
                       submissions=submissions)
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        top1 = f"{summary['holdout_top1']:.2%}" if "holdout_top1" in summary else "–"
        log(f"== {run_name} finished: holdout top-1 {top1} | trainable {params['trainable_excl_head']:,} "
            f"({params['trainable_pct']:.3f}%) | {summary['gpu_hours']:.2f} GPU-h")
        return summary
    finally:
        LOG_FILE = None


def predict_only(run_name):
    """Phase 2 用：加载训练好的 run，只做预测，提交文件写回该 run 的文件夹"""
    global LOG_FILE
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    with open(os.path.join(run_dir, "config.json")) as f:
        saved = json.load(f)
    cfg = resolve_config({k: saved[k] for k in ARCH_KEYS if k in saved})
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        free_gpu()
        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        model = apply_method(model, cfg).to(DEVICE)
        ck = torch.load(os.path.join(run_dir, "final.pt"), map_location="cpu", weights_only=False)
        _, unexpected = model.load_state_dict(ck["state"], strict=False)
        assert not unexpected, f"unexpected keys: {unexpected[:5]}"
        log(f"Loaded trained weights from {run_dir}/final.pt")
        if cfg.METHOD == "lora":
            model = model.merge_and_unload()
        summary_path = os.path.join(run_dir, "summary.json")
        summary = {"run": run_name, "method": cfg.METHOD}
        if os.path.exists(summary_path):
            with open(summary_path) as f:
                summary = json.load(f)
        summary.update(latency_report(model, ctx, cfg))
        summary.setdefault("submissions", {}).update(
            {split: make_submission(split, items, model, ctx, cfg, run_name, run_dir) for split, items in PRED_ITEMS.items()})
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        return summary
    finally:
        LOG_FILE = None

## 10. 运行实验队列

按 `EXPERIMENTS` 的顺序依次运行。如果零样本基线的本地验证准确率低于 25%（文档报告约 43.7%），说明数据管线很可能有问题，队列会停下来，避免浪费几个小时的 GPU。

In [ ]:
RESULTS = []
if LOAD_RUN:
    RESULTS.append(predict_only(LOAD_RUN))
else:
    for i, exp in enumerate(EXPERIMENTS, 1):
        exp = dict(exp)
        name = exp.pop("RUN_NAME", None) or f"{BATCH_NAME}_{i}_{exp.get('METHOD', METHOD)}"
        result = run_experiment(exp, name)
        RESULTS.append(result)
        if result["method"] == "linear" and result.get("proto_top1") is not None and result["proto_top1"] < 0.25:
            raise RuntimeError(f"Zero-shot holdout top-1 is only {result['proto_top1']:.1%} (expected ~43.7%): "
                               "the data pipeline is probably broken, stopping the queue. Please check the log.")

## 11. 自动最终训练（可选，默认关闭）

只有 `RUN_FINAL = True` 时才会运行：在已训练的实验里（不含 `linear`）选本地验证 top-1 最高的方法（或 `FINAL_METHOD` 指定的方法），沿用它的配置，再套用 `FINAL_OVERRIDES`（默认 15 个 epoch、用满每类 45 张）。它不会根据短实验结果调整学习率、轮数等参数，所以更推荐看完短实验结果后，把定好的最终配置写进 `EXPERIMENTS` 另跑一批。

In [ ]:
if RUN_FINAL and not LOAD_RUN:
    trained = [r for r in RESULTS if r.get("steps") and r.get("holdout_top1") is not None]
    ranking = ", ".join(f"{r['method']} {r['holdout_top1']:.2%}" for r in sorted(trained, key=lambda r: -r["holdout_top1"]))
    if FINAL_METHOD:
        pick = next((r for r in trained if r["method"] == FINAL_METHOD), None)
        base = dict(pick["overrides"]) if pick else {"METHOD": FINAL_METHOD}
    else:
        base = dict(max(trained, key=lambda r: r["holdout_top1"])["overrides"]) if trained else None
    if base is None:
        log("RUN_FINAL: no trained experiment with holdout accuracy to choose from; skipping the final run")
    else:
        final_overrides = {**base, **FINAL_OVERRIDES}
        log(f"\nFinal run with {final_overrides}  (holdout ranking: {ranking or 'n/a'})")
        RESULTS.append(run_experiment(final_overrides, f"{BATCH_NAME}_final_{final_overrides.get('METHOD', METHOD)}"))

## 12. 结果对比（写报告用）

In [ ]:
def fmt_pct(v):
    return f"{v:.2%}" if isinstance(v, (int, float)) else "–"


header = ("| Run | Method | Holdout top-1 | Holdout top-5 | Trainable params (excl. head) | Classifier head | "
          "Latency bs1 (ms) | GPU-hours | Epochs |\n|---|---|---|---|---|---|---|---|---|")
rows = [f"| {r['run']} | {r['method']} | {fmt_pct(r.get('holdout_top1'))} | {fmt_pct(r.get('holdout_top5'))} | "
        f"{r.get('trainable_excl_head', 0):,} ({r.get('trainable_pct', 0):.3f}%) | {r.get('head_mode', '–')} | "
        f"{r.get('latency_ms_bs1', float('nan')):.2f} | {r.get('gpu_hours', 0):.2f} | {r.get('epochs_trained', 0)} |"
        for r in RESULTS]
table = header + "\n" + "\n".join(rows)
with open(os.path.join(BATCH_DIR, "comparison.md"), "w") as f:
    f.write(table + "\n")
with open(os.path.join(BATCH_DIR, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)
log("\n" + table)
try:
    from IPython.display import Markdown, display
    display(Markdown(table))
except ImportError:
    pass

# 对比图：每个训练过的实验一条线（最多 4 条），横轴统一用 epoch
curves = []
for r in RESULTS:
    path = os.path.join(OUT_ROOT, "runs", r["run"], "history.json")
    if r.get("steps") and os.path.exists(path):
        with open(path) as f:
            curves.append((r["run"].replace(f"{BATCH_NAME}_", ""), json.load(f)))
if len(curves) > len(SERIES):
    log(f"Comparison plot shows the first {len(SERIES)} trained runs only")
if curves:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=SURFACE)
    ends = []
    for (label, h), color in zip(curves, SERIES):
        spe = h.get("steps_per_epoch") or 1
        axes[0].plot([s / spe for s in h["step"]], h["loss"], color=color, linewidth=1.5, label=label)
        acc = [(e, 100 * a) for e, a in zip(h["epoch"], h["holdout_top1"]) if a is not None]
        if acc:
            e, a = zip(*acc)
            axes[1].plot(e, a, color=color, linewidth=1.5, marker="o", markersize=6,
                         markeredgecolor=SURFACE, markeredgewidth=1.5, label=label)
            ends.append((a[-1], e[-1]))
    # 末端数值标签：按数值排序后上下错开，避免重叠
    for rank, (value, epoch) in enumerate(sorted(ends, reverse=True)):
        axes[1].annotate(f"{value:.1f}%", (epoch, value), textcoords="offset points",
                         xytext=(6, 10 * (len(ends) - 1) / 2 - 10 * rank), va="center", color=INK, fontsize=9)
    style_axis(axes[0], "Training loss", "epoch", "cross-entropy")
    style_axis(axes[1], "Holdout top-1 accuracy", "epoch", "accuracy (%)")
    axes[1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(frameon=False, labelcolor=INK, fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(BATCH_DIR, "comparison.png"), dpi=150, facecolor=SURFACE)
    plt.show()

## 13. 自动断开（可选）

结果此时已经全部存进 Drive。`DISCONNECT_WHEN_DONE = True` 会在确认数据写入 Drive 后释放运行时，停止消耗计算单元；同一个会话里还要接着跑实验时不要打开。

In [ ]:
if IN_COLAB and DISCONNECT_WHEN_DONE:
    from google.colab import drive, runtime
    log("All done; releasing the runtime ...")
    drive.flush_and_unmount()  # 确认所有结果都已上传到 Drive
    runtime.unassign()

## 下一步

- **提交**：每个实验的 `submission_val_*.zip` 在 Drive 的 `cs604/runs/<run 名>/` 里，下载后上传到 CodaBench（每天最多 5 次），记下分数。
- **完整训练**：根据短实验结果定好最终参数后，把它写进 `EXPERIMENTS` 另跑一批，例如 `EXPERIMENTS = [dict(METHOD="full", EPOCHS=15, HOLDOUT_PER_CLASS=0)]`（超出时间预算会自动缩短；`HOLDOUT_PER_CLASS=0` 表示用满每类 45 张，此时准确率要看 CodaBench）。
- **写报告**：`cs604/batches/<BATCH_NAME>/` 里的 `comparison.md` 和 `comparison.png` 可以直接放进报告；每个 run 文件夹里的 `train_log.txt` 就是要附的训练日志。
- **消融实验**：改 `EXPERIMENTS` 就能排一组新的实验，比如分类头初始化方式（`HEAD_INIT` = random / proto / lp）、LoRA 的 rank 和插入位置（`LORA_R`、`LORA_TARGETS`）、Adapter 维度、学习率等。例：`dict(METHOD="lora", EPOCHS=3, WARMUP_EPOCHS=0.3, LORA_R=16, LORA_TARGETS=["qkv", "proj", "fc1", "fc2"])`。
- **Phase 2（10/31 发布测试集）**：先再运行一次 `cs604_download_to_drive.ipynb`，把测试集存进 Drive；然后设 `LOAD_RUN = "<最终训练的 run 名>"`、`PREDICT_SPLITS = ["test"]`，再「全部运行」即可生成 `test_team_XX.txt`。不需要重新训练，也不会下载训练集。